In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import CarSim
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)

In [ ]:
class Mission2(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=80)
        self.goals = [
            GoalCircle((2, 0.0), 0.2, should_stop=False),
            GoalCircle((4.3, 1.0), 0.2, should_stop=False),
            GoalCircle((3.2, 2.2), 0.2, should_stop=False),
            GoalCircle((1.7, 1.0), 0.2, should_stop=False),
            GoalCircle((0.0, 0.8), 0.2, should_stop=False),
            GoalCircle((0.0, 0.0), 0.2),
        ]
        self.initial_xy = (0.0, 0.0)
        self.random_d_xy = (0.0, 0.1)
        self.random_d_yaw_deg = 1
        self.set_signs(
            [
                ### 標識を変更するにはここから下を書き換える
                Sign(x=1.1, y=-0.1, name="left"),
                Sign(x=2.0, y=-0.1, name="left"),
                Sign(x=2.9, y=-0.1, name="left"),
                Sign(x=3.7, y=0.1, name="left"),
                Sign(x=4.3, y=0.8, name="left"),
                Sign(x=4.3, y=1.6, name="left"),
                Sign(x=3.7, y=2.3, name="left"),
                Sign(x=2.8, y=2.2, name="left"),
                Sign(x=3.1, y=1.2, name="right"),
                Sign(x=2.3, y=1.0, name="right"),
                Sign(x=1.4, y=1.0, name="left"),
                Sign(x=0.5, y=0.9, name="left"),
                Sign(x=-0.2, y=0.8, name="left"),
                Sign(x=-0.4, y=0.2, name="left"),
                Sign(x=0.2, y=-0.2, name="stop"),
                ### 標識を変更するにはここより上を書き換える
            ]
        )

    @staticmethod
    def command_func(alive, *, move, rotate, search, **kwargs):
        """回答例：
        回転と直進を使用して標識に近づく
        - leftを最後に見た後に標識を見失ったら反時計回りに回転する
        - rightを最後に見た後に標識を見失ったら時計回りに回転する
        """
        ######## ここから下にプログラムを書こう
        last_name = None
        while alive():
            pos = search()
            if pos is None:
                if last_name == "left":
                    rotate(w=45)
                elif last_name == "right":
                    rotate(w=-45)
                else:
                    move(v=0)
            else:
                last_name = pos.name
                if pos.theta > 5:
                    rotate(w=45)
                elif pos.theta < -5:
                    rotate(w=-45)
                else:
                    move(v=0.2)
        ######## ここより上にプログラムを書こう


sim = CarSim(prop, Mission2())
sim.run()
SimDrawer(sim).show()

In [ ]:
# 複数回実行して必ず成功するかのチェック
CHECK_CNT = 10
ok = 0
for i in range(CHECK_CNT):
    mission = Mission2()
    sim = CarSim(prop, mission)
    complete = sim.run()
    if complete:
        goal_cnt = sim.history.goal_cnt[-1]
        goals = len(mission.goals)
        goaled = goal_cnt == goals
        if goaled:
            ok += 1
        else:
            SimDrawer(sim).show()
            break
        print("-------------------------------OK:", ok)